## Testing TMBd access token 

In [1]:
import os
import requests
from dotenv import load_dotenv

load_dotenv("../.env")

TMDB_TOKEN = os.getenv("TMDB_READ_ACCESS_TOKEN")
TMDB_API_KEY = os.getenv("TMDB_API_KEY")

if not TMDB_TOKEN:
    raise ValueError("TMDB_READ_ACCESS_TOKEN is missing")

if not TMDB_API_KEY:
    raise ValueError("TMDB_API_KEY is missing")

print("TMDB tokens loaded successfully.")

TMDB tokens loaded successfully.


## Use a test id to check if the request can return success on API call and print json formart

In [2]:
imdb_id = "tt0133093"  # temporary test ID

url = f"https://api.themoviedb.org/3/find/{imdb_id}"

headers = {
    "Authorization": f"Bearer {TMDB_TOKEN}",
    "accept": "application/json",
}

params = {
    "external_source": "imdb_id"
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30,
)

print("Status:", response.status_code)

data = response.json()
data

Status: 200


{'movie_results': [{'adult': False,
   'backdrop_path': '/lrtSb1skJayPydZk0OSMAKjBOVe.jpg',
   'id': 603,
   'title': 'The Matrix',
   'original_title': 'The Matrix',
   'overview': 'Set in the 22nd century, The Matrix tells the story of a computer hacker who joins a group of underground insurgents fighting the vast and powerful computers who now rule the earth.',
   'poster_path': '/aOIuZAjPaRIE6CMzbazvcHuHXDc.jpg',
   'media_type': 'movie',
   'original_language': 'en',
   'genre_ids': [28, 878],
   'popularity': 50.5808,
   'release_date': '1999-03-31',
   'softcore': False,
   'video': False,
   'vote_average': 8.258,
   'vote_count': 28721}],
 'person_results': [],
 'tv_results': [],
 'tv_episode_results': [],
 'tv_season_results': []}

In [3]:
movie = data["movie_results"][0]

poster_path = movie["poster_path"]

poster_url = f"https://image.tmdb.org/t/p/w500{poster_path}"

print("Title:", movie["title"])
print("Poster path:", poster_path)
print("Poster URL:", poster_url)

Title: The Matrix
Poster path: /aOIuZAjPaRIE6CMzbazvcHuHXDc.jpg
Poster URL: https://image.tmdb.org/t/p/w500/aOIuZAjPaRIE6CMzbazvcHuHXDc.jpg


In [4]:
test_titles = [
    ("dt0907_001", "tt0075314"),
    ("dt0907_002", "tt0068473"),
    ("dt0907_003", "tt0071853"),
    ("dt0907_004", "tt0061578"),
    ("dt0907_005", "tt0063929"),
    ("dt0907_006", "tt0079470"),
    ("dt0907_007", "tt0066999"),
    ("dt0907_008", "tt0061418"),
    ("dt0907_009", "tt0080453"),
    ("dt0907_010", "tt0054953"),
]

results = []

for title_id, imdb_id in test_titles:
    url = f"https://api.themoviedb.org/3/find/{imdb_id}"

    response = requests.get(
        url,
        headers=headers,
        params={"external_source": "imdb_id"},
        timeout=30,
    )

    data = response.json()

    # IMDb IDs can resolve to either a movie or TV show.
    matches = data.get("movie_results", []) or data.get("tv_results", [])

    if matches:
        match = matches[0]
        poster_path = match.get("poster_path")

        poster_url = (
            f"https://image.tmdb.org/t/p/w500{poster_path}"
            if poster_path
            else None
        )

        results.append({
            "title_id": title_id,
            "imdb_id": imdb_id,
            "tmdb_id": match.get("id"),
            "tmdb_title": match.get("title") or match.get("name"),
            "poster_url": poster_url,
            "status": "MATCHED",
        })
    else:
        results.append({
            "title_id": title_id,
            "imdb_id": imdb_id,
            "tmdb_id": None,
            "tmdb_title": None,
            "poster_url": None,
            "status": "NOT_FOUND",
        })

In [5]:
import pandas as pd

poster_test_df = pd.DataFrame(results)

poster_test_df

,title_id,imdb_id,tmdb_id,tmdb_title,poster_url,status
0,dt0907_001,tt0075314,103,Taxi Driver,https://image.tmdb.org/t/p/w500/ekstpH614fwDX8...,MATCHED
1,dt0907_002,tt0068473,10669,Deliverance,https://image.tmdb.org/t/p/w500/2TrAzNJlHyNYYS...,MATCHED
2,dt0907_003,tt0071853,762,Monty Python and the Holy Grail,https://image.tmdb.org/t/p/w500/7nTkHjETdGMYK1...,MATCHED
3,dt0907_004,tt0061578,1654,The Dirty Dozen,https://image.tmdb.org/t/p/w500/tFWWsuhp22zJ6O...,MATCHED
4,dt0907_005,tt0063929,849,Monty Python's Flying Circus,https://image.tmdb.org/t/p/w500/hZtMtL8Pqv9FAM...,MATCHED
5,dt0907_006,tt0079470,583,Life of Brian,https://image.tmdb.org/t/p/w500/lSSA64WF0M0BXn...,MATCHED
6,dt0907_007,tt0066999,984,Dirty Harry,https://image.tmdb.org/t/p/w500/scl2JDHzYoIEs5...,MATCHED
7,dt0907_008,tt0061418,475,Bonnie and Clyde,https://image.tmdb.org/t/p/w500/sCSQFK9kMsprT4...,MATCHED
8,dt0907_009,tt0080453,5689,The Blue Lagoon,https://image.tmdb.org/t/p/w500/hg57Zr63A96mpI...,MATCHED
9,dt0907_010,tt0054953,10911,The Guns of Navarone,https://image.tmdb.org/t/p/w500/j6VSFnm20GlkUi...,MATCHED


In [6]:
import sys

from pathlib import Path


# The notebook lives inside /notebooks, so the project root is one level above.
project_root = Path.cwd().parent


if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# Display the detected root so we know the notebook is running from the correct project.
print("Project root:", project_root)

Project root: /Users/mac/Documents/netflix-data-engineering


In [7]:
from dotenv import load_dotenv
from pathlib import Path

env_path = Path.cwd().parent / ".env"

load_dotenv(env_path, override=True)

print("Loaded:", env_path)

Loaded: /Users/mac/Documents/netflix-data-engineering/.env


In [8]:
from src.database import get_etl_connection

conn = get_etl_connection()

print("Database connection successful")

conn.close()

Database connection successful


In [9]:
from src.database import get_etl_connection
from src.poster_enrichment import enrich_posters

conn = get_etl_connection()

stats = enrich_posters(conn, limit=10)

print(stats)

conn.close()

{'processed': 0, 'matched': 0, 'not_found': 0, 'errors': 0}


In [10]:
from src.database import get_etl_connection
from src.poster_enrichment import enrich_posters

conn = get_etl_connection()

stats = enrich_posters(conn, limit=100)

print(stats)

conn.close()

{'processed': 0, 'matched': 0, 'not_found': 0, 'errors': 0}


In [11]:
from src.database import get_etl_connection
from src.poster_enrichment import enrich_posters

conn = get_etl_connection()

try:
    stats = enrich_posters(conn, limit=500)
    print(stats)
finally:
    conn.close()

{'processed': 0, 'matched': 0, 'not_found': 0, 'errors': 0}


In [12]:
from src.database import get_etl_connection
from src.poster_enrichment import enrich_posters

conn = get_etl_connection()

try:
    stats = enrich_posters(conn, limit=1000)
    print(stats)
finally:
    conn.close()

{'processed': 0, 'matched': 0, 'not_found': 0, 'errors': 0}


In [13]:
from src.database import get_etl_connection
from src.poster_enrichment import enrich_posters

conn = get_etl_connection()

try:
    stats = enrich_posters(conn, limit=1000)
    print(stats)
finally:
    conn.close()

{'processed': 0, 'matched': 0, 'not_found': 0, 'errors': 0}


In [14]:
from src.database import get_etl_connection
from src.poster_enrichment import enrich_posters

conn = get_etl_connection()

try:
    stats = enrich_posters(conn, limit=1000)
    print(stats)
finally:
    conn.close()

{'processed': 0, 'matched': 0, 'not_found': 0, 'errors': 0}


In [15]:
from src.database import get_etl_connection
from src.poster_enrichment import enrich_posters

conn = get_etl_connection()

try:
    stats = enrich_posters(conn, limit=1000)
    print(stats)
finally:
    conn.close()

{'processed': 0, 'matched': 0, 'not_found': 0, 'errors': 0}


In [16]:
from src.database import get_etl_connection
from src.poster_enrichment import enrich_posters

conn = get_etl_connection()

try:
    stats = enrich_posters(conn, limit=1000)
    print(stats)
finally:
    conn.close()

{'processed': 0, 'matched': 0, 'not_found': 0, 'errors': 0}


In [17]:
from src.database import get_etl_connection
from src.poster_enrichment import enrich_posters

conn = get_etl_connection()

try:
    stats = enrich_posters(conn, limit=1000)
    print(stats)
finally:
    conn.close()

{'processed': 0, 'matched': 0, 'not_found': 0, 'errors': 0}


In [18]:
import os
import requests

TMDB_API_KEY = os.getenv("TMDB_API_KEY")
TMDB_BASE_URL = "https://api.themoviedb.org/3"

candidates = [
    (3681, "ts236033", "#blackAF", "SHOW", 2020),
    (1080, "tm132675", "100 Things to Do Before High School", "MOVIE", 2014),
    (4669, "tm979073", "100% Halal", "MOVIE", 2020),
    (4618, "tm852207", "3 Logical Exits", "MOVIE", 2020),
    (5451, "tm993790", "40 Love", "MOVIE", 2021),
    (2911, "tm835357", "A 3 Minute Hug", "MOVIE", 2018),
    (5112, "tm1044275", "A Classic Horror Story", "MOVIE", 2021),
    (2972, "tm321563", "A Drowning Man", "MOVIE", 2017),
    (5122, "tm1201670", "A Farewell to Ozark", "MOVIE", 2022),
    (4499, "tm918959", "A Go! Go! Cory Carson Summer Camp", "MOVIE", 2020),
]

In [19]:
import os
import requests

TMDB_API_KEY = os.getenv("TMDB_API_KEY")
TMDB_BASE_URL = "https://api.themoviedb.org/3"

candidates = [
    (3681, "ts236033", "#blackAF", "SHOW", 2020),
    (1080, "tm132675", "100 Things to Do Before High School", "MOVIE", 2014),
    (4669, "tm979073", "100% Halal", "MOVIE", 2020),
    (4618, "tm852207", "3 Logical Exits", "MOVIE", 2020),
    (5451, "tm993790", "40 Love", "MOVIE", 2021),
    (2911, "tm835357", "A 3 Minute Hug", "MOVIE", 2018),
    (5112, "tm1044275", "A Classic Horror Story", "MOVIE", 2021),
    (2972, "tm321563", "A Drowning Man", "MOVIE", 2017),
    (5122, "tm1201670", "A Farewell to Ozark", "MOVIE", 2022),
    (4499, "tm918959", "A Go! Go! Cory Carson Summer Camp", "MOVIE", 2020),
]

In [20]:
def search_tmdb_fallback(title, content_type, release_year):
    media_type = "tv" if content_type == "SHOW" else "movie"

    params = {
        "api_key": TMDB_API_KEY,
        "query": title,
    }

    if media_type == "movie":
        params["year"] = release_year
    else:
        params["first_air_date_year"] = release_year

    response = requests.get(
        f"{TMDB_BASE_URL}/search/{media_type}",
        params=params,
        timeout=30,
    )

    response.raise_for_status()

    return response.json().get("results", [])

In [21]:
print(search_tmdb_fallback)

<function search_tmdb_fallback at 0x1180820e0>


In [22]:
test = search_tmdb_fallback(
    "#blackAF",
    "SHOW",
    2020
)

print(len(test))

if test:
    print(test[0])

1
{'adult': False, 'backdrop_path': '/sSRFTSdVbpcsZV6332MzBbGky5G.jpg', 'genre_ids': [35], 'id': 101200, 'origin_country': ['US'], 'original_language': 'en', 'original_name': '#blackAF', 'overview': 'A father takes an irreverent and honest approach to parenting and relationships.', 'popularity': 4.3225, 'poster_path': '/rZsWeJzJbsb14IjNgTeBP3FEmvx.jpg', 'first_air_date': '2020-04-17', 'softcore': False, 'name': '#blackAF', 'vote_average': 5.94, 'vote_count': 42}


In [23]:
fallback_results = []

for title_sk, title_id, title, content_type, release_year in candidates:

    results = search_tmdb_fallback(
        title,
        content_type,
        release_year,
    )

    if results:
        best = results[0]

        tmdb_title = (
            best.get("name")
            if content_type == "SHOW"
            else best.get("title")
        )

        tmdb_date = (
            best.get("first_air_date")
            if content_type == "SHOW"
            else best.get("release_date")
        )

        fallback_results.append({
            "title_sk": title_sk,
            "source_title": title,
            "source_year": release_year,
            "type": content_type,
            "tmdb_id": best.get("id"),
            "tmdb_title": tmdb_title,
            "tmdb_date": tmdb_date,
            "poster_path": best.get("poster_path"),
            "result_count": len(results),
        })

    else:
        fallback_results.append({
            "title_sk": title_sk,
            "source_title": title,
            "source_year": release_year,
            "type": content_type,
            "tmdb_id": None,
            "tmdb_title": None,
            "tmdb_date": None,
            "poster_path": None,
            "result_count": 0,
        })

In [24]:
import pandas as pd

fallback_df = pd.DataFrame(fallback_results)

fallback_df[
    [
        "source_title",
        "source_year",
        "type",
        "tmdb_title",
        "tmdb_date",
        "poster_path",
        "result_count",
    ]
]

,source_title,source_year,type,tmdb_title,tmdb_date,poster_path,result_count
0,#blackAF,2020,SHOW,#blackAF,2020-04-17,/rZsWeJzJbsb14IjNgTeBP3FEmvx.jpg,1
1,100 Things to Do Before High School,2014,MOVIE,100 Things to Do Before High School,2014-11-10,/4yvsG2b5cv9T06UvKUEIUsDvizd.jpg,1
2,100% Halal,2020,MOVIE,100% Halal,2020-12-01,/9ePxnVy1oyK68kQ3AgBE3R3GSeZ.jpg,1
3,3 Logical Exits,2020,MOVIE,3 Logical Exits,2020-01-22,/uI01wITFbKD6GFBrN7ux3YGpdIH.jpg,1
4,40 Love,2021,MOVIE,40-Love,2021-10-29,/2Ld3CWyyQh1XMoWACSqRtSLN3l5.jpg,1
5,A 3 Minute Hug,2018,MOVIE,A 3 Minute Hug,2018-11-20,/n9V1ijGKFwcho4M2PfFlIWvHgxp.jpg,1
6,A Classic Horror Story,2021,MOVIE,A Classic Horror Story,2021-07-01,/bifYmCujLshZo4qVdmUpP7Q2iLn.jpg,1
7,A Drowning Man,2017,MOVIE,A Drowning Man,2017-05-17,/hv7EyiPceNEXGZmx2aUwKKFGN9Q.jpg,1
8,A Farewell to Ozark,2022,MOVIE,A Farewell to Ozark,2022-04-29,/wNxnVljephNMyLKK9j4jRuByRFG.jpg,1
9,A Go! Go! Cory Carson Summer Camp,2020,MOVIE,A Go! Go! Cory Carson Summer Camp,2020-07-04,/bcJgqj1iylRYUwSqg2NCfuhN2K2.jpg,1


In [26]:
from src.database import get_etl_connection
from src.poster_enrichment import enrich_missing_posters

conn = get_etl_connection()

try:
    stats = enrich_missing_posters(conn, limit=10)
    print(stats)
finally:
    conn.close()

{'processed': 10, 'fallback_matched': 9, 'still_not_found': 1, 'errors': 0}


In [27]:
from src.database import get_etl_connection
from src.poster_enrichment import enrich_missing_posters

conn = get_etl_connection()

try:
    stats = enrich_missing_posters(conn)
    print(stats)
finally:
    conn.close()

{'processed': 306, 'fallback_matched': 202, 'still_not_found': 104, 'errors': 0}
